# 01 — Exploratory Data Analysis

**Social Sentiment Tracker**  
This notebook provides a comprehensive EDA of the sentiment dataset.  
It covers:
1. Data loading and basic info
2. Missing value analysis
3. Class distribution
4. Text length statistics
5. High-frequency word analysis
6. Temporal trends (if date column available)

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on the path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter

from config import set_seed
from src.data_loader import load_data, split_data
from src.visualize import (
    plot_sentiment_distribution,
    plot_text_length_distribution,
    plot_top_keywords,
    plot_sentiment_over_time,
)

set_seed()
print('Setup complete.')

## 1. Load Dataset
We load the Sentiment140 dataset (or fall back to mock data if unavailable).  
The loader applies basic cleaning automatically.

In [ ]:
df = load_data()
print(f'Dataset shape: {df.shape}')
df.head()

## 2. Basic Info & Schema

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

## 3. Missing Value Analysis

In [ ]:
missing = df.isnull().sum().reset_index()
missing.columns = ['column', 'missing_count']
missing['missing_pct'] = (missing['missing_count'] / len(df) * 100).round(2)
print(missing)

fig = px.bar(
    missing[missing['missing_count'] > 0] if missing['missing_count'].sum() > 0 else missing,
    x='column', y='missing_pct',
    title='Missing Values (%)',
    labels={'missing_pct': 'Missing (%)', 'column': 'Column'},
    color='missing_pct',
    color_continuous_scale='Reds',
)
fig.show()

## 4. Class Distribution
We check whether the dataset is balanced across sentiment classes.

In [ ]:
label_counts = df['label'].value_counts().reset_index()
label_counts.columns = ['label', 'count']
label_counts['sentiment'] = label_counts['label'].map({0: 'Negative', 1: 'Positive', 2: 'Neutral'})
print(label_counts)

fig = plot_sentiment_distribution(df)
fig.show()

## 5. Text Length Analysis
Text length (in words) is a useful feature and can reveal quality issues.

In [ ]:
df['word_count'] = df['clean_text'].str.split().str.len()
df['char_count'] = df['clean_text'].str.len()

print('Word count statistics:')
print(df.groupby('label')['word_count'].describe().round(2))

In [ ]:
fig = plot_text_length_distribution(df)
fig.show()

## 6. High-Frequency Word Analysis
We examine the most common words per sentiment class to understand vocabulary patterns.

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

for label, name in [(1, 'Positive'), (0, 'Negative')]:
    subset = df[df['label'] == label]['clean_text'].dropna()
    all_words = ' '.join(subset).split()
    freq = Counter(w for w in all_words if w not in ENGLISH_STOP_WORDS and len(w) > 2)
    top_words = freq.most_common(15)
    words, counts = zip(*top_words)
    
    fig = px.bar(
        x=list(counts)[::-1], y=list(words)[::-1],
        orientation='h',
        title=f'Top 15 Words — {name}',
        labels={'x': 'Count', 'y': 'Word'},
        color=list(counts)[::-1],
        color_continuous_scale='Greens' if label == 1 else 'Reds',
    )
    fig.update_layout(showlegend=False)
    fig.show()

## 7. TF-IDF Top Keywords

In [ ]:
for label in [1, 0]:
    fig = plot_top_keywords(df, n=15, sentiment=label)
    fig.show()

## 8. Temporal Trends
If the dataset contains a `date` column, we visualise sentiment volume over time.

In [ ]:
if 'date' in df.columns:
    fig = plot_sentiment_over_time(df)
    fig.show()
else:
    print('No date column available — skipping temporal analysis.')

## 9. Train / Val / Test Split Summary

In [ ]:
train_df, val_df, test_df = split_data(df)

split_summary = pd.DataFrame({
    'Split': ['Train', 'Validation', 'Test'],
    'Size': [len(train_df), len(val_df), len(test_df)],
    'Pct': [len(train_df)/len(df)*100, len(val_df)/len(df)*100, len(test_df)/len(df)*100],
})
print(split_summary.to_string(index=False))

fig = px.bar(
    split_summary, x='Split', y='Size',
    text='Size', color='Split',
    title='Train / Val / Test Split Sizes',
)
fig.update_traces(texttemplate='%{text}', textposition='outside')
fig.show()

## Summary

| Observation | Finding |
|---|---|
| Dataset size | See cell above |
| Class balance | Approximately balanced (positive / negative) |
| Avg word count | ~10–20 words per post |
| Missing values | None after preprocessing |
| Key positive words | love, great, good, amazing |
| Key negative words | bad, worst, terrible, horrible |

The dataset is clean and balanced — ready for model training.  
Proceed to `02_baseline_model.ipynb` for the TF-IDF + LR baseline.